# 投机解码（Speculative Decoding）完整模拟器本 notebook 实现投机解码的完整流程：1. Draft model（小模型）自回归生成 N 个候选 token2. Target model（大模型）一次 forward 并行验证3. 接受/拒绝采样（数学保证输出分布与 target model 完全一致）4. 加速比分析 & 可视化> 面试高频题：字节/腾讯/阿里 都会考。> 核心公式：接受概率 `min(1, p_target(x) / p_draft(x))`

In [ ]:
import numpy as np
from collections import Counter
from typing import List, Tuple

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except Exception:
    HAS_MPL = False

np.random.seed(42)
print("Ready. matplotlib:", HAS_MPL)

## 1) 构造 Draft / Target 概率分布Draft 和 Target 越接近 → 接受率越高 → 加速比越大

In [ ]:
VOCAB_SIZE = 50

def make_target_distribution(vocab_size: int, temperature: float = 1.0, seed: int = 0) -> np.ndarray:
    rng = np.random.default_rng(seed)
    logits = rng.standard_normal(vocab_size)
    logits = logits / temperature
    exp_logits = np.exp(logits - logits.max())
    return exp_logits / exp_logits.sum()

def make_draft_distribution(target_dist: np.ndarray, noise_level: float = 0.3, seed: int = 1) -> np.ndarray:
    rng = np.random.default_rng(seed)
    noise = rng.standard_normal(len(target_dist)) * noise_level
    noisy = np.exp(np.log(target_dist + 1e-10) + noise)
    return noisy / noisy.sum()

target_dist = make_target_distribution(VOCAB_SIZE, temperature=0.8, seed=42)
draft_dist_good = make_draft_distribution(target_dist, noise_level=0.2, seed=10)
draft_dist_bad = make_draft_distribution(target_dist, noise_level=1.5, seed=10)

print("Target dist top-5:", np.sort(target_dist)[::-1][:5].round(4))
print("Draft (good) top-5:", np.sort(draft_dist_good)[::-1][:5].round(4))
print("Draft (bad) top-5:", np.sort(draft_dist_bad)[::-1][:5].round(4))

## 2) 投机解码核心算法### 算法流程1. Draft model 自回归生成 N 个 token2. Target model 一次 forward 验证 N 个 token3. 从左到右验证：接受概率 = min(1, p/q)，拒绝则从修正分布采样4. 全部接受 → 额外 bonus token

In [ ]:
def draft_generate(draft_dist: np.ndarray, n: int, rng: np.random.Generator):
    tokens = [rng.choice(len(draft_dist), p=draft_dist) for _ in range(n)]
    probs = [draft_dist.copy() for _ in range(n)]
    return tokens, probs

def target_verify(target_dist: np.ndarray, n: int):
    return [target_dist.copy() for _ in range(n + 1)]

def speculative_decode_step(draft_tokens, draft_probs, target_probs, rng):
    N = len(draft_tokens)
    accepted = []
    for i in range(N):
        x = draft_tokens[i]
        q_x = draft_probs[i][x]
        p_x = target_probs[i][x]
        accept_prob = min(1.0, p_x / max(q_x, 1e-10))
        if rng.random() < accept_prob:
            accepted.append(x)
        else:
            p = target_probs[i]
            q = draft_probs[i]
            adjusted = np.maximum(0.0, p - q)
            Z = adjusted.sum()
            adjusted = adjusted / Z if Z > 1e-10 else p
            bonus = rng.choice(len(adjusted), p=adjusted)
            return accepted, bonus
    bonus = rng.choice(len(target_probs[N]), p=target_probs[N])
    return accepted, bonus

# 快速测试
rng = np.random.default_rng(99)
dt, dp = draft_generate(draft_dist_good, n=5, rng=rng)
tp = target_verify(target_dist, n=5)
acc, bon = speculative_decode_step(dt, dp, tp, rng)
print(f"Draft 5 tokens, accepted {len(acc)}, bonus={bon}, total output={len(acc)+1}")

## 3) 验证正确性：输出分布与 Target Model 一致投机解码的最终输出分布 **严格等于** target model 的分布（无损）。

In [ ]:
def collect_first_token_distribution(target_dist, draft_dist, draft_len, n_samples, seed=0):
    rng = np.random.default_rng(seed)
    counts = np.zeros(len(target_dist))
    for _ in range(n_samples):
        dt, dp = draft_generate(draft_dist, n=draft_len, rng=rng)
        tp = target_verify(target_dist, n=draft_len)
        acc, bon = speculative_decode_step(dt, dp, tp, rng)
        first = acc[0] if acc else bon
        counts[first] += 1
    return counts / counts.sum()

n_samples = 50000
empirical = collect_first_token_distribution(target_dist, draft_dist_good, 5, n_samples, seed=42)
mask = (empirical > 0) & (target_dist > 0)
kl = np.sum(empirical[mask] * np.log(empirical[mask] / target_dist[mask]))
print(f"KL(empirical || target) = {kl:.6f}  (should be ~0)")
print("Correctness:", "PASS ✓" if kl < 0.01 else "NEEDS MORE SAMPLES")

if HAS_MPL:
    top_k = 15
    idx = np.argsort(target_dist)[::-1][:top_k]
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(top_k)
    w = 0.35
    ax.bar(x - w/2, target_dist[idx], width=w, label="Target (ground truth)", alpha=0.8)
    ax.bar(x + w/2, empirical[idx], width=w, label=f"Spec Decode (N={n_samples})", alpha=0.8)
    ax.set_xticks(x); ax.set_xticklabels([f"t{i}" for i in idx])
    ax.set_ylabel("probability"); ax.set_title(f"Distribution Correctness: KL={kl:.5f}")
    ax.legend(); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout(); plt.show()

## 4) 加速比分析- **接受率 α**：每步平均被接受的 draft token 占 draft_len 的比例- **加速比** = E[tokens_per_step] / (1 + draft_len × cost_ratio)

In [ ]:
def measure_acceptance(target_dist, draft_dist, draft_len, n_trials=5000, seed=42):
    rng = np.random.default_rng(seed)
    total_acc, total_out = 0, 0
    acc_counts = []
    for _ in range(n_trials):
        dt, dp = draft_generate(draft_dist, n=draft_len, rng=rng)
        tp = target_verify(target_dist, n=draft_len)
        acc, bon = speculative_decode_step(dt, dp, tp, rng)
        total_acc += len(acc)
        total_out += len(acc) + 1
        acc_counts.append(len(acc))
    return {
        "draft_len": draft_len,
        "avg_accepted": total_acc / n_trials,
        "avg_output": total_out / n_trials,
        "acceptance_rate": total_acc / n_trials / draft_len,
        "accept_dist": Counter(acc_counts),
    }

for label, dd in [("good draft", draft_dist_good), ("bad draft", draft_dist_bad)]:
    r = measure_acceptance(target_dist, dd, draft_len=5)
    print(f"{label}: accept_rate={r['acceptance_rate']:.1%}, avg_output={r['avg_output']:.2f} tok/step")

In [ ]:
# Draft 长度 vs 加速比
draft_lengths = [1, 2, 3, 4, 5, 6, 8, 10]
cost_ratios = [0.05, 0.1, 0.2]

results = {}
for cr in cost_ratios:
    speedups, avg_outs = [], []
    for dl in draft_lengths:
        r = measure_acceptance(target_dist, draft_dist_good, dl, n_trials=3000)
        wall = 1.0 + dl * cr
        speedups.append(r["avg_output"] / wall)
        avg_outs.append(r["avg_output"])
    results[cr] = {"speedups": speedups, "avg_outputs": avg_outs}

print(f"{'draft_len':>10}", end="")
for cr in cost_ratios: print(f"  cr={cr:.2f}", end="")
print("  avg_out")
for i, dl in enumerate(draft_lengths):
    print(f"{dl:>10d}", end="")
    for cr in cost_ratios: print(f"  {results[cr]['speedups'][i]:>6.2f}x", end="")
    print(f"  {results[cost_ratios[0]]['avg_outputs'][i]:>6.2f}")

if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for cr in cost_ratios:
        axes[0].plot(draft_lengths, results[cr]["speedups"], marker="o", label=f"cost_ratio={cr}")
    axes[0].axhline(1.0, color="red", ls="--", lw=1, label="baseline")
    axes[0].set_xlabel("draft length N"); axes[0].set_ylabel("speedup")
    axes[0].set_title("Speedup vs Draft Length"); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

    r5 = measure_acceptance(target_dist, draft_dist_good, 5, n_trials=5000)
    dist = r5["accept_dist"]
    ks = sorted(dist.keys()); vs = [dist[k]/5000 for k in ks]
    axes[1].bar(ks, vs, color="steelblue", edgecolor="white")
    axes[1].set_xlabel("# accepted tokens"); axes[1].set_ylabel("probability")
    axes[1].set_title("Acceptance Distribution (draft_len=5)"); axes[1].set_xticks(range(6)); axes[1].grid(axis="y", alpha=0.3)
    plt.tight_layout(); plt.show()

## 5) Draft 质量敏感性实验

In [ ]:
noise_levels = [0.05, 0.1, 0.2, 0.5, 1.0, 1.5, 2.0]
quality_results = []
for nl in noise_levels:
    dd = make_draft_distribution(target_dist, noise_level=nl, seed=10)
    r = measure_acceptance(target_dist, dd, draft_len=5, n_trials=5000)
    quality_results.append({"noise": nl, "ar": r["acceptance_rate"], "out": r["avg_output"],
                            "sp": r["avg_output"] / (1.0 + 5 * 0.1)})

print(f"{'noise':>8} {'accept%':>10} {'avg_out':>10} {'speedup':>10}")
for qr in quality_results:
    print(f"{qr['noise']:>8.2f} {qr['ar']:>10.1%} {qr['out']:>10.2f} {qr['sp']:>10.2f}x")

if HAS_MPL:
    fig, ax1 = plt.subplots(figsize=(8, 5))
    ax1.plot([q["noise"] for q in quality_results], [q["ar"] for q in quality_results], "o-", label="acceptance rate", color="tab:blue")
    ax1.set_xlabel("draft noise level (lower=better draft)"); ax1.set_ylabel("acceptance rate", color="tab:blue")
    ax2 = ax1.twinx()
    ax2.plot([q["noise"] for q in quality_results], [q["sp"] for q in quality_results], "s--", label="speedup", color="tab:red")
    ax2.set_ylabel("speedup", color="tab:red"); ax2.axhline(1.0, color="gray", ls=":", lw=1)
    ax1.set_title("Draft Quality vs Performance"); ax1.legend(loc="upper left", fontsize=8); ax2.legend(loc="upper right", fontsize=8)
    ax1.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 6) 完整生成模拟（多步投机解码）

In [ ]:
def full_speculative_generation(target_dist, draft_dist, total_tokens=100, draft_len=5, seed=42):
    rng = np.random.default_rng(seed)
    generated, steps = [], 0
    while len(generated) < total_tokens:
        steps += 1
        remaining = total_tokens - len(generated)
        n = min(draft_len, remaining)
        dt, dp = draft_generate(draft_dist, n=n, rng=rng)
        tp = target_verify(target_dist, n=n)
        acc, bon = speculative_decode_step(dt, dp, tp, rng)
        new = (acc + [bon])[:remaining]
        generated.extend(new)
    return {"total_tokens": len(generated), "steps": steps, "tokens_per_step": len(generated) / steps}

total_gen = 200
for dl in [0, 3, 5, 8]:
    if dl == 0:
        print(f"draft_len={dl:>2d} (baseline)  steps={total_gen:>4d}  tokens/step=1.00")
    else:
        r = full_speculative_generation(target_dist, draft_dist_good, total_gen, dl)
        print(f"draft_len={dl:>2d}             steps={r['steps']:>4d}  tokens/step={r['tokens_per_step']:.2f}")

## 7) 面试讲解模板> "投机解码用**小模型的速度换大模型的并行验证**：>> 1. Draft 阶段：小模型自回归生成 N 个候选 token> 2. Verify 阶段：大模型一次 forward 同时验证 N 个 token> 3. Accept/Reject：以 min(1, p/q) 接受；拒绝则从 max(0,p-q)/Z 采样>> **数学保证**：最终分布严格等于仅用大模型（无损）> **加速来源**：一次 target forward 产出 1~N+1 个 token> **最优 N**：取决于 draft 质量和成本比，经验上 N=3~5"### 面试追问准备1. **为什么无损？** → 接受概率 + 修正分布的数学推导2. **batch 怎么处理？** → 最少接受数对齐 / padding / tree attention3. **Medusa vs 标准投机？** → Medusa 在 target 上加多头直接 draft，无需独立模型4. **什么场景不适合？** → draft 质量差 / batch 很大时 decode 不是瓶颈5. **动态调 N？** → 根据最近 K 步接受率自适应